# nb_03b — Gold : `fact_workforce_event` avec résolution de clé as-of

Nous attribuons chaque événement de définition de rémunération à
la **grille salariale en vigueur à la date de l’événement**, puis calculons un **compa-ratio** que nous pouvons
comparer à la grille recalibrée **d’aujourd’hui**.

Résolution pour chaque événement :
- `date_key` → `dim_date`
- `cost_center_key` → `dim_cost_center` (SCD1, direct)
- `worker_key` → `dim_worker` **as-of** `event_date` (jointure de plage SCD2)
- `pay_band_key` → `dim_pay_band` **as-of** sur (group, level, event_date) (jointure de plage SCD2)

Nous précalculons les bornes de tranche et le compa-ratio au niveau de l’événement afin que le DAX du modèle reste simple.

## Charger les entrées de la table de faits

**Résumé.** Charge les événements Silver et les dimensions, en donnant un alias aux colonnes de validité SCD2 afin que les jointures de plage as-of qui suivent soient lisibles.

<details>
<summary>Détails ligne par ligne</summary>

- `ev = spark.table("silver.workforce_event")` — les événements conformes à charger.
- `dcc` — recherche de clé du centre de coûts (SCD1, jointure directe sur `cost_center_id`).
- `dw` — versions des travailleurs avec `effective_from`/`effective_to` renommés en alias `w_from`/`w_to` pour la jointure as-of.
- `dpb` — versions des tranches salariales avec le groupe et le niveau renommés en alias `pb_group`/`pb_level`, ainsi que les bornes de tranche et la plage de validité.
- `print(...)` — indique le nombre d’événements qui seront chargés.

</details>

In [ ]:
from pyspark.sql import functions as F
ev  = spark.table("silver.workforce_event")
dcc = spark.table("gold.dim_cost_center").select("cost_center_key","cost_center_id")
dw  = spark.table("gold.dim_worker").select("worker_key","employee_id",
        F.col("effective_from").alias("w_from"), F.col("effective_to").alias("w_to"))
dpb = spark.table("gold.dim_pay_band").select("pay_band_key",
        F.col("classification_group").alias("pb_group"),
        F.col("classification_level").alias("pb_level"),
        "band_min","band_mid","band_max","effective_from","effective_to")
print(f"silver events to load: {ev.count():,}")

## Construire la table de faits avec résolution de clé as-of

**Résumé.** Résout les clés de dimension de chaque événement — le centre de coûts directement, le travailleur et la tranche salariale **as-of** de la date de l’événement via des jointures de plage SCD2 — puis précalcule le salaire de base, la prime, le compa-ratio et l’indicateur de salaire inférieur à la tranche.

<details>
<summary>Détails ligne par ligne</summary>

- `date_key` — la jointure entière `yyyyMMdd` vers `dim_date`.
- `.join(dcc, "cost_center_id", "left")` — clé directe du centre de coûts SCD1.
- Jointure de plage du travailleur : `employee_id` égal **et** `event_date` compris entre `w_from` et `w_to` — sélectionne la version du travailleur en vigueur lors de l’événement.
- Jointure de plage de la tranche salariale : le groupe et le niveau sont égaux **et** `event_date` est compris entre `effective_from`/`effective_to` de la tranche — sélectionne la grille en vigueur lors de l’événement.
- `base_salary_cad` — le montant en CAD uniquement pour les événements de rémunération (`Hire`, `Promotion`, `Step Increment`).
- `bonus_cad` — le montant en CAD uniquement pour `Performance Pay`.
- `compa_ratio_at_event` — `base_salary_cad / band_mid` (arrondi) lorsqu’un salaire de base et un point médian positif existent.
- `below_band_at_event` — indique si le salaire de base est inférieur au minimum de la tranche.
- Le `.select(...)` définit les colonnes finales (bornes de tranche renommées avec l’alias `*_at_event`) ; `write ... saveAsTable("gold.fact_workforce_event")` et `print` enregistrent les données et indiquent le nombre de lignes.

</details>

In [ ]:
COMP = ["Hire","Promotion","Step Increment"]



fact = (ev

    .withColumn("date_key", F.date_format("event_date","yyyyMMdd").cast("int"))

    .join(dcc, "cost_center_id", "left")

    # jointure de plage worker selon la date (SCD2)

    .join(dw, (ev.employee_id==dw.employee_id) &

              (ev.event_date>=dw.w_from) & (ev.event_date<=dw.w_to), "left")

    # jointure de plage pay band selon le groupe, le niveau et la date (SCD2)

    .join(dpb, (ev.classification_group==dpb.pb_group) &

               (ev.classification_level==dpb.pb_level) &

               (ev.event_date>=dpb.effective_from) &

               (ev.event_date<=dpb.effective_to), "left")

    # mesures

    .withColumn("base_salary_cad",

        F.when(F.col("event_type").isin(COMP), F.col("amount_cad")))

    .withColumn("bonus_cad",

        F.when(F.col("event_type")=="Performance Pay", F.col("amount_cad")))

    .withColumn("compa_ratio_at_event",

        F.when(F.col("base_salary_cad").isNotNull() & (F.col("band_mid")>0),

               F.round(F.col("base_salary_cad")/F.col("band_mid"),4)))

    .withColumn("below_band_at_event",

        F.when(F.col("base_salary_cad").isNotNull(),

               F.col("base_salary_cad") < F.col("band_min")))

    .select("event_id","date_key","cost_center_key","worker_key","pay_band_key",

            "classification_group","classification_level","event_type",

            "amount_cad","base_salary_cad","bonus_cad",

            F.col("band_min").alias("band_min_at_event"),

            F.col("band_mid").alias("band_mid_at_event"),

            F.col("band_max").alias("band_max_at_event"),

            "compa_ratio_at_event","below_band_at_event",

            "local_currency","source_system","ingest_ts"))



(fact.write.format("delta").mode("overwrite").option("overwriteSchema","true")

    .saveAsTable("gold.fact_workforce_event"))



print(f"fact rows: {fact.count():,}")


## Contrôle de qualité des données — aucune clé non résolue

**Résumé.** Compte les lignes pour lesquelles une clé de dimension n’a pas été résolue, afin qu’une jointure défaillante soit signalée immédiatement au lieu de corrompre silencieusement l’analyse.

<details>
<summary>Détails ligne par ligne</summary>

- La requête `SELECT sum(case when ... is null ...)` compte les valeurs nulles de `worker_key`, `pay_band_key` et `cost_center_key`, ainsi que le nombre total de lignes. Tous les comptes de valeurs nulles doivent être égaux à zéro.

</details>

In [ ]:
spark.sql("""
  SELECT sum(case when worker_key    is null then 1 else 0 end) AS null_worker,
         sum(case when pay_band_key  is null then 1 else 0 end) AS null_pay_band,
         sum(case when cost_center_key is null then 1 else 0 end) AS null_cost_center,
         count(*) AS total
  FROM gold.fact_workforce_event""").show()

## Compa-ratio as-was par rapport à as-is

Compa-ratio moyen de la rémunération définie en 2021, mesuré par rapport à la grille **en vigueur à cette date**
(as-was) — toutes ces rémunérations paraissent saines, proches de 1,0. Mais la grille a depuis été
recalibrée à la hausse, de sorte que les mêmes salaires se situent plus bas par rapport à la grille **d’aujourd’hui**.

<details>
<summary>Détails ligne par ligne</summary>

- La requête joint la table de faits à `dim_date`, filtre les lignes avec un `base_salary_cad`, puis les regroupe par `d.year` (l’année où la rémunération a été définie).
- `avg(f.compa_ratio_at_event)` — le compa-ratio as-was moyen par année.
- `sum(case when f.below_band_at_event then 1 else 0 end)` — le nombre as-was de rémunérations inférieures à la tranche par année.

</details>

In [ ]:
spark.sql("""
  SELECT d.year AS pay_set_year,
         round(avg(f.compa_ratio_at_event),3) AS avg_compa_ratio_as_was,
         sum(case when f.below_band_at_event then 1 else 0 end) AS count_below_band_as_was
  FROM gold.fact_workforce_event f
  JOIN gold.dim_date d ON f.date_key = d.date_key
  WHERE f.base_salary_cad IS NOT NULL
  GROUP BY d.year ORDER BY d.year""").show()